In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("iceberg-notebook")
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.demo.type", "rest")
    .config("spark.sql.catalog.demo.uri", "http://rest:8181")
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.demo.warehouse", "s3://warehouse/wh/")
    .config("spark.sql.catalog.demo.s3.endpoint", "http://seaweedfs:8333")
    .config("spark.sql.catalog.demo.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", "demo")
    .getOrCreate()
)

print("Spark session initialized with the SeaweedFS-backed Iceberg catalog.")

In [ ]:
%%sql

CREATE DATABASE IF NOT EXISTS bronze;

In [ ]:
%%sql 

CREATE DATABASE IF NOT EXISTS silver;


In [ ]:
%%sql 

CREATE DATABASE IF NOT EXISTS gold;

In [ ]:
%%sql
CREATE NAMESPACE IF NOT EXISTS bronze;

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for user information'
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7,2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (category)
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for product items'
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.purchases (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12,2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.pageviews (
    user_id BIGINT,
    url STRING,
    channel STRING,
    received_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(received_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
);


In [ ]:
%%sql
CREATE NAMESPACE IF NOT EXISTS silver;

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS silver.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    valid_email BOOLEAN,
    full_name STRING
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Validated dimension table for user information'
);

In [ ]:
%%sql
    
CREATE TABLE IF NOT EXISTS silver.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7,2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (category)
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for product items'
);

In [ ]:
%%sql
    
CREATE TABLE IF NOT EXISTS silver.purchases_enriched (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12,2),
    total_price DECIMAL(14,2),         
    user_email STRING,                
    item_name STRING,
    item_category STRING,
    purchase_date DATE,
    purchase_hour INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP                  
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Validated and enriched fact table for purchase transactions'
);

In [ ]:
table_name = "demo.silver.purchases_enriched"
existing_columns = {
    field.name.lower() for field in spark.table(table_name).schema.fields
}
new_columns = []
if "purchase_date" not in existing_columns:
    new_columns.append("purchase_date DATE")
if "purchase_hour" not in existing_columns:
    new_columns.append("purchase_hour INT")

if new_columns:
    spark.sql(
        f"ALTER TABLE {table_name} ADD COLUMNS ({', '.join(new_columns)})"
    )
    print(f"Added columns to {table_name}: {', '.join(new_columns)}")
else:
    print(f"{table_name} already has purchase_date and purchase_hour")

In [ ]:
%%sql
    
CREATE TABLE IF NOT EXISTS silver.pageviews_by_items (
    user_id BIGINT,
    item_id BIGINT,
    page STRING,
    item_name STRING,
    item_category STRING,
    channel STRING,
    received_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(received_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
);

In [ ]:
%%sql
SHOW DATABASES;

In [ ]:
%%sql
show tables in bronze;

In [ ]:
%%sql
show tables in silver;

In [ ]:
%%sql
DESCRIBE TABLE bronze.users;

In [ ]:
%%sql
SELECT * FROM silver.purchases_enriched.properties;

In [ ]:
%%sql
SHOW CATALOGS;

In [ ]:
%%sql
SHOW TABLES IN demo.silver;

In [ ]:
%%sql
SHOW TBLPROPERTIES demo.silver.purchases_enriched;

In [ ]:
%%sql
DESCRIBE TABLE EXTENDED demo.silver.purchases_enriched;

In [ ]:
%%sql
SELECT * FROM demo.silver.purchases_enriched LIMIT 10;

In [ ]:
%%sql
DESCRIBE TABLE silver.pageviews_by_items;